# 10. Construcción del catálogo de artículos

**Entregable 4:** obtener un archivo con `article_id`, título, autores, año, área, keywords, abstract, DOI y URL.

Se conserva `indice` como `article_id`. Cada registro del catálogo corresponde a una manifestación bibliográfica y reúne los autores UNAM presentes en la base. El orden de autores se conserva según el CSV; no se asume que sea el orden editorial. Los textos se copian completos y los campos vacíos permanecen vacíos.

## 1. Leer la base final

Se utiliza el CSV como entrada única. La segunda ruta permite ejecutar también cuando el directorio de trabajo es la raíz del proyecto. `dtype=str` conserva el texto, y `keep_default_na=False` mantiene los vacíos como cadenas vacías en lugar de convertirlos a `NaN`.

In [1]:
import pandas as pd

# Rutas relativas: funciona desde notebooks/ o desde la raíz del proyecto.
carpeta = "../"
entrada = "04_Limpieza/06_Finales/Canonico_FINAL.csv"

try:
    base = pd.read_csv(carpeta + entrada, dtype=str,
                       keep_default_na=False, encoding="utf-8-sig")
except FileNotFoundError:
    carpeta = ""
    base = pd.read_csv(entrada, dtype=str,
                       keep_default_na=False, encoding="utf-8-sig")

print(f"Filas leídas: {len(base):,}")

Filas leídas: 1,151


## 2. Comprobar los datos

Antes de seleccionar un valor se comprueba que cada índice tenga los mismos metadatos en todas sus filas. Un vacío frente a un valor presente también se señala como diferencia. Si aparece un error, revisa el caso antes de continuar; el código no elige automáticamente entre valores contradictorios.

In [2]:
# Los metadatos deben coincidir en todas las filas del mismo artículo.
metadatos = ["Titulo", "Año", "Area", "Keywords", "Abstract", "Doi", "URL"]
requeridas = ["indice", "Autor_norm"] + metadatos
faltantes = [col for col in requeridas if col not in base.columns]

if faltantes:
    raise ValueError(f"Faltan columnas: {faltantes}")
if base.empty:
    raise ValueError("El CSV no contiene registros.")

for col in ["indice", "Año", "Autor_norm"]:
    if base[col].str.strip().eq("").any():
        raise ValueError(f"Hay valores vacíos en {col}.")

# Mantener el identificador y el año como números enteros.
base = base.astype({"indice": "int64", "Año": "int64"})

variantes = base.groupby("indice")[metadatos].nunique(dropna=False)
conflictos = variantes.index[(variantes > 1).any(axis=1)].tolist()

if conflictos:
    raise ValueError(f"Revisar metadatos diferentes en los índices: {conflictos}")

print("Validación correcta: los metadatos coinciden por artículo.")

Validación correcta: los metadatos coinciden por artículo.


## 3. Reunir las publicaciones y sus autores

La agrupación se realiza por `indice`, no por título. El identificador se conserva con el nombre `article_id`, y los autores se separan con punto y coma. La vista previa muestra cinco campos para facilitar la lectura; el catálogo completo contiene las nueve columnas, incluidos los resúmenes íntegros.

In [3]:
# Una fila por índice; autores UNAM reunidos sin repetir nombres.
# "first" es seguro aquí porque los metadatos ya se comprobaron.
catalogo = base.groupby("indice", as_index=False, sort=True).agg(
    titulo=("Titulo", "first"),
    autores=("Autor_norm", lambda nombres: "; ".join(nombres.drop_duplicates())),
    anio=("Año", "first"),
    area=("Area", "first"),
    keywords=("Keywords", "first"),
    abstract=("Abstract", "first"),
    doi=("Doi", "first"),
    url=("URL", "first")
).rename(columns={"indice": "article_id"})

print(f"Catálogo: {len(catalogo):,} registros y {len(catalogo.columns)} columnas.")
catalogo[["article_id", "titulo", "autores", "anio", "area"]].head()

Catálogo: 519 registros y 9 columnas.


,article_id,titulo,autores,anio,area
0,1,A comparative study of NeuralODE and Universal...,Raymundo Vazquez Martinez,2024,CC
1,2,A Comparative Study of Numerical Laplace Inver...,Vladimir Martínez Bernardino,2025,CC
2,3,A Greenfield Analysis for Supply Chains Enhanc...,Jose Antonio Marmolejo Saucedo,2024,CC
3,4,A Hierarchical Approach to a Tri-Objective Por...,Carlos Ignacio Hernández Castellanos; Yeudiel ...,2024,CC
4,5,A hybrid framework for assessing regional iner...,Mario Roberto Arrieta Paternina,2025,CC


## 4. Guardar CSV y Excel

Se generan los dos archivos a partir del mismo catálogo, sin agregar una columna de índice de pandas. El Excel contiene una hoja llamada `Articulos`.

In [4]:
# La carpeta 05_grafos ya debe existir en el proyecto.
salida = carpeta + "05_grafos/catalogo_articulos"

catalogo.to_csv(salida + ".csv", index=False, encoding="utf-8-sig")
catalogo.to_excel(salida + ".xlsx", index=False,
                  sheet_name="Articulos", engine="openpyxl")

print("Archivos guardados:")
print(salida + ".csv")
print(salida + ".xlsx")

Archivos guardados:
../05_grafos/catalogo_articulos.csv
../05_grafos/catalogo_articulos.xlsx
